# Building and training the model

In [29]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    cross_validate,
    KFold
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

import joblib

In [30]:
df = pd.read_csv("training_dataset.csv")

train = df[df["ratio_enfants"].notna()].copy()

predict = df[df["ratio_enfants"].isna()].copy()

In [31]:
train = train.replace([np.inf, -np.inf], np.nan)

train = train.dropna().reset_index(drop=True)

In [32]:
features = [
    "population",
    "longitude",
    "latitude"
]

X = train[features]

In [33]:
targets = [

    "ratio_enfants",

    "ratio_jeunes",

    "ratio_actifs",

    "ratio_seniors",

    "ratio_femmes"

]

Y = train[targets]

In [34]:
X_train, X_test, Y_train, Y_test = train_test_split(

    X,

    Y,

    test_size=0.2,

    random_state=42

)

In [35]:
models = {

    "Random Forest":

        MultiOutputRegressor(

            RandomForestRegressor(

                n_estimators=300,

                random_state=42,

                n_jobs=-1

            )

        ),

    "XGBoost":

        MultiOutputRegressor(

            XGBRegressor(

                n_estimators=300,

                random_state=42

            )

        ),

    "LightGBM":

        MultiOutputRegressor(

            LGBMRegressor(

                n_estimators=300,

                random_state=42

            )

        ),

    "CatBoost":

        MultiOutputRegressor(

            CatBoostRegressor(

                iterations=300,

                verbose=False,

                random_state=42

            )

        )

}

In [36]:
results = []

best_model = None
best_r2 = -999

for name, model in models.items():

    print("="*60)

    print(name)

    model.fit(X_train, Y_train)

    pred = model.predict(X_test)

    mae = mean_absolute_error(Y_test, pred)

    rmse = np.sqrt(mean_squared_error(Y_test, pred))

    r2 = r2_score(

        Y_test,

        pred,

        multioutput="uniform_average"

    )

    print("MAE :", mae)

    print("RMSE :", rmse)

    print("R² :", r2)

    results.append({

        "Model":name,

        "MAE":mae,

        "RMSE":rmse,

        "R2":r2

    })

    if r2 > best_r2:

        best_r2 = r2

        best_model = model

Random Forest
MAE : 0.0015137886276197312
RMSE : 0.00408756300899501
R² : 0.9706549616281406
XGBoost
MAE : 0.003978297580033541
RMSE : 0.006680229299061978
R² : 0.9237732887268066
LightGBM
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000427 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 95015, number of used features: 3
[LightGBM] [Info] Start training from score 0.262286
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000249 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 95015, number of used features: 3
[LightGBM] [Info] Start training from score 0.167859
[Li

In [42]:
results = pd.DataFrame(results)

results.sort_values(

    "R2",

    ascending=False

)

,Model,MAE,RMSE,R2
0,Random Forest,0.001514,0.004088,0.970655
1,XGBoost,0.003978,0.006680,0.923773
2,LightGBM,0.005650,0.008350,0.880050
3,CatBoost,0.006779,0.009802,0.834850


In [43]:
joblib.dump(

    best_model,

    "best_demography_model.pkl"

)

['best_demography_model.pkl']

In [44]:
X_missing = predict[features]

predictions = best_model.predict(X_missing)

predict[targets] = predictions

In [45]:
df.loc[predict.index, targets] = predict[targets]

In [46]:
df.to_csv(

    "demography.csv",

    index=False

)